# Module 14: Python for Machine Learning — Solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, fetch_california_housing, load_digits
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay, confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, r2_score
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Setup complete')

### Solution 1: Linear Regression

In [ ]:
housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_h_train_s = scaler.fit_transform(X_h_train)
X_h_test_s = scaler.transform(X_h_test)

lr = LinearRegression()
lr.fit(X_h_train_s, y_h_train)
y_pred = lr.predict(X_h_test_s)

print('=== Linear Regression Results ===')
print(f'MSE: {mean_squared_error(y_h_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_h_test, y_pred)):.4f}')
print(f'MAE: {np.mean(np.abs(y_h_test - y_pred)):.4f}')
print(f'R2: {r2_score(y_h_test, y_pred):.4f}')

coef_df = pd.DataFrame({'feature': housing.feature_names, 'coef': lr.coef_})
print('\nTop positive:', coef_df.nlargest(3, 'coef').to_string(index=False))
print('Top negative:', coef_df.nsmallest(3, 'coef').to_string(index=False))

### Solution 2: Decision Tree on Iris

In [ ]:
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_i_train, X_i_test, y_i_train, y_i_test = train_test_split(X_i, y_i, test_size=0.2, random_state=42)

for depth in [2, 4, 6]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_i_train, y_i_train)
    acc = accuracy_score(y_i_test, dt.predict(X_i_test))
    print(f'max_depth={depth}: Accuracy={acc:.4f}')

# Best tree visualization
dt_best = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_i_train, y_i_train)
plt.figure(figsize=(15, 8))
plot_tree(dt_best, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.title('Decision Tree (max_depth=4)')
plt.show()

y_pred = dt_best.predict(X_i_test)
ConfusionMatrixDisplay.from_predictions(y_i_test, y_pred, display_labels=iris.target_names)
plt.title('Confusion Matrix')
plt.show()

### Solution 3: Cross-Validation Comparison

In [ ]:
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].dropna()
titanic['sex'] = (titanic['sex'] == 'male').astype(int)
titanic = pd.get_dummies(titanic, columns=['embarked'], drop_first=True)
X_t, y_t = titanic.drop('survived', axis=1), titanic['survived']

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_t, y_t, cv=5, scoring='accuracy')
    cv_results[name] = {'mean': scores.mean(), 'std': scores.std(), 'scores': scores}
    print(f'{name}: mean={scores.mean():.4f}, std={scores.std():.4f}')

best_model = max(cv_results, key=lambda k: cv_results[k]['mean'])
print(f'\nBest model: {best_model}')

### Solution 4: ROC-AUC Analysis

In [ ]:
X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(X_t, y_t, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_t_train_s = scaler.fit_transform(X_t_train)
X_t_test_s = scaler.transform(X_t_test)

lr = LogisticRegression(max_iter=500, random_state=42).fit(X_t_train_s, y_t_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_t_train_s, y_t_train)

fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(lr, X_t_test_s, y_t_test, ax=ax, name='Logistic Regression')
RocCurveDisplay.from_estimator(rf, X_t_test_s, y_t_test, ax=ax, name='Random Forest')
plt.title('ROC Curves on Titanic')
plt.show()

### Solution 5: SVM Kernels

In [ ]:
# Binary: setosa (1) vs rest (0)
y_binary = (y_i == 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X_i, y_binary, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for kernel in ['linear', 'rbf', 'poly']:
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test_s))
    print(f'SVM ({kernel}): Accuracy={acc:.4f}')

### Solution 6: K-Means Elbow Method

In [ ]:
# Use MedInc, Latitude, Longitude
X_cluster = housing.data[['MedInc', 'Latitude', 'Longitude']]
scaler = StandardScaler()
X_cluster_s = scaler.fit_transform(X_cluster)

inertias = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_s)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()
print('Optimal k appears to be around 3-4 based on the elbow.')

### Solution 7: PCA Visualization

In [ ]:
X_iris = iris.data
y_iris = iris.target

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_iris, cmap='viridis', s=60, alpha=0.8)
plt.colorbar(scatter, label='Species')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('PCA of Iris Dataset')
plt.show()
print(f'Total variance explained: {pca.explained_variance_ratio_.sum():.1%}')

### Solution 8: Hyperparameter Tuning

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1],
    'kernel': ['rbf', 'linear']
}

svm = SVC(random_state=42)
grid = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_t_train_s, y_t_train)

print('=== SVM GridSearchCV Results ===')
print(f'Best parameters: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')
print(f'Test accuracy: {accuracy_score(y_t_test, grid.predict(X_t_test_s)):.4f}')

### Solution 9: Feature Importance

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_t_train_s, y_t_train)
importances = pd.DataFrame({
    'feature': X_t.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=importances, x='importance', y='feature')
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()
print('Top 3 features:')
print(importances.head(3).to_string(index=False))

### Solution 10: Full ML Workflow

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Find best k via CV
k_scores = []
for k in [3, 5, 7]:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s, y_train, cv=5)
    k_scores.append((k, scores.mean()))

best_k = max(k_scores, key=lambda x: x[1])[0]
print(f'Best k from CV: {best_k}')

# Train and evaluate
knn = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.4f}')

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=iris.target_names)
plt.title(f'KNN (k={best_k}) Confusion Matrix')
plt.show()

# Decision boundary via PCA
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_s)
X_test_2d = pca.transform(X_test_s)
knn_2d = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_2d, y_train)

xx, yy = np.meshgrid(np.linspace(X_train_2d[:,0].min()-1, X_train_2d[:,0].max()+1, 100),
                     np.linspace(X_train_2d[:,1].min()-1, X_train_2d[:,1].max()+1, 100))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X_test_2d[:, 0], X_test_2d[:, 1], c=y_test, cmap='viridis', edgecolors='k', s=60)
plt.title(f'KNN (k={best_k}) Decision Boundary (PCA-reduced)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()
print('Complete ML workflow executed successfully.')